In [38]:
import pandas as pd
import pyarrow.parquet as pq
import requests
import os
from pathlib import Path


In [39]:
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/warehouse").mkdir(parents=True, exist_ok=True)
Path("data/quarantine").mkdir(parents=True, exist_ok=True)

In [40]:
import requests

PARQUET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"


response = requests.get(PARQUET_URL)
with open("data/raw/yellow_2023_01.parquet", "wb") as f:
    f.write(response.content)

response = requests.get(ZONE_URL)
with open("data/raw/taxi_zones.csv", "wb") as f:
    f.write(response.content)

print("Files downloaded successfully")


Files downloaded successfully


In [41]:
df = pd.read_parquet("data/raw/yellow_2023_01.parquet")


In [42]:

df.head(3)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0


In [43]:
df.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
dtype: object

In [44]:
df = df.rename(columns={
    'VendorID'              : 'vendor_id',
    'tpep_pickup_datetime'  : 'pickup_datetime',
    'tpep_dropoff_datetime' : 'dropoff_datetime',
    'passenger_count'       : 'passenger_count',
    'trip_distance'         : 'trip_distance_miles',
    'PULocationID'          : 'pickup_location_id',
    'DOLocationID'          : 'dropoff_location_id',
    'RatecodeID'            : 'rate_code_id',
    'payment_type'          : 'payment_type',
    'fare_amount'           : 'fare_amount',
    'extra'                 : 'extra_charges',
    'mta_tax'               : 'mta_tax',
    'tip_amount'            : 'tip_amount',
    'tolls_amount'          : 'tolls_amount',
    'total_amount'          : 'total_amount',
})
keep_cols = [
    'vendor_id', 'pickup_datetime', 'dropoff_datetime',
    'passenger_count', 'trip_distance_miles', 'pickup_location_id',
    'dropoff_location_id', 'payment_type', 'fare_amount',
    'tip_amount', 'total_amount', 'extra_charges', 'mta_tax', 'tolls_amount'
]
df = df[[c for c in keep_cols if c in df.columns]]

In [45]:
df['pickup_datetime']  = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

df['pickup_date']=df['pickup_datetime'].dt.date
df['pickup_year']=df['pickup_datetime'].dt.year
df['pickup_month']=df['pickup_datetime'].dt.month
df['pickup_day']=df['pickup_datetime'].dt.day
df['pickup_hour']=df['pickup_datetime'].dt.hour
df['pickup_weekday'] = df['pickup_datetime'].dt.day_name()

df['trip_duration_sec']=(df['dropoff_datetime']-df['pickup_datetime']).dt.total_seconds()


df['avg_speed_mph'] = df.apply(lambda row: round(row['trip_distance_miles'] / (row['trip_duration_sec'] / 3600), 2)
    if row['trip_duration_sec'] > 0 else None,
    axis=1
)
df.insert(0, 'trip_id', range(1, len(df) + 1))
df[['trip_id','pickup_datetime','trip_duration_sec','avg_speed_mph']].head(3)

,trip_id,pickup_datetime,trip_duration_sec,avg_speed_mph
0,1,2023-01-01 00:32:10,506.0,6.90
1,2,2023-01-01 00:55:08,379.0,10.45
2,3,2023-01-01 00:25:04,765.0,11.81


In [46]:
rules = {
    'zero_or_negative_duration': df['trip_duration_sec'] <= 0,
    'negative_fare': df['fare_amount'] < 0,
    'negative_distance': df['trip_distance_miles'] < 0,
    'impossible_speed': df['avg_speed_mph'] > 200,
    'zero_passengers': df['passenger_count'] <= 0,
    'dropoff_before_pickup': df['dropoff_datetime'] < df['pickup_datetime'],
    'missing_location_id': df['pickup_location_id'].isna() | df['dropoff_location_id'].isna(),
}

is_bad = rules['zero_or_negative_duration']
for condition in rules.values():
    is_bad = is_bad | condition


clean_df=df[~is_bad].copy()
quarantine_df=df[is_bad].copy()

# Print report
print("Data Quality Report")
print("=" * 40)
print(f"Total records : {len(df):>10,}")
print(f"Clean records : {len(clean_df):>10,}  ({len(clean_df)/len(df)*100:.1f}%)")
print(f"Bad records   : {len(quarantine_df):>10,}  ({len(quarantine_df)/len(df)*100:.1f}%)")
print()
print("Per-Rule Failure Count:")
for rule_name, condition in rules.items():
    count = condition.sum()
    print(f"{rule_name:<35}:{count:,}")

Data Quality Report
Total records :  3,066,766
Clean records :  2,988,459  (97.4%)
Bad records   :     78,307  (2.6%)

Per-Rule Failure Count:
zero_or_negative_duration          :1,121
negative_fare                      :25,049
negative_distance                  :0
impossible_speed                   :1,237
zero_passengers                    :51,164
dropoff_before_pickup              :3
missing_location_id                :0


In [47]:

quarantine_df.to_csv("data/quarantine/bad_trips.csv", index=False)
print(f"Bad records saved to: data/quarantine/bad_trips.csv")
quarantine_df[['trip_id','fare_amount','trip_duration_sec','avg_speed_mph','passenger_count']].head(5)

Bad records saved to: data/quarantine/bad_trips.csv


,trip_id,fare_amount,trip_duration_sec,avg_speed_mph,passenger_count
3,4,12.1,577.0,11.85,0.0
132,133,-5.1,154.0,9.82,1.0
263,264,-9.3,458.0,9.35,2.0
324,325,-25.4,1301.0,13.53,1.0
485,486,8.6,359.0,13.04,0.0


In [48]:
dim_geography = pd.read_csv("data/raw/taxi_zones.csv")
dim_geography.columns = ['location_id', 'borough', 'zone', 'service_zone']
print(dim_geography.shape)
dim_geography.head()

(265, 4)


,location_id,borough,zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [49]:
# ── dim_date ── (unique dates from clean trips)
dim_date = (
    clean_df[['pickup_date','pickup_year','pickup_month','pickup_day','pickup_weekday']]
    .drop_duplicates(subset='pickup_date')
    .reset_index(drop=True)
)
dim_date['is_weekend'] = dim_date['pickup_weekday'].isin(['Saturday','Sunday'])
dim_date['date_id']    = dim_date['pickup_date'].astype(str).str.replace('-','').astype(int)

print("dim_date:")
print(f" Shape: {dim_date.shape}")
dim_date.head(5)

dim_date:
 Shape: (36, 7)


,pickup_date,pickup_year,pickup_month,pickup_day,pickup_weekday,is_weekend,date_id
0,2023-01-01,2023,1,1,Sunday,True,20230101
1,2022-12-31,2022,12,31,Saturday,True,20221231
2,2022-10-24,2022,10,24,Monday,False,20221024
3,2022-10-25,2022,10,25,Tuesday,False,20221025
4,2023-01-02,2023,1,2,Monday,False,20230102


In [50]:
fact_trips = clean_df[[
    'trip_id',
    'pickup_date',          
    'pickup_year',           
    'pickup_month',          
    'pickup_hour',
    'vendor_id',            
    'pickup_location_id',    
    'dropoff_location_id',   
    'payment_type',        
    'passenger_count',
    'trip_distance_miles',
    'trip_duration_sec',
    'avg_speed_mph',
    'fare_amount',
    'tip_amount',
    'total_amount',
    'mta_tax',
    'tolls_amount',
    'extra_charges',
]].copy()
print(fact_trips.shape)
fact_trips.head(3)

(2988459, 19)


,trip_id,pickup_date,pickup_year,pickup_month,pickup_hour,vendor_id,pickup_location_id,dropoff_location_id,payment_type,passenger_count,trip_distance_miles,trip_duration_sec,avg_speed_mph,fare_amount,tip_amount,total_amount,mta_tax,tolls_amount,extra_charges
0,1,2023-01-01,2023,1,0,2,161,141,2,1.0,0.97,506.0,6.90,9.3,0.0,14.3,0.5,0.0,1.0
1,2,2023-01-01,2023,1,0,2,43,237,1,1.0,1.10,379.0,10.45,7.9,4.0,16.9,0.5,0.0,1.0
2,3,2023-01-01,2023,1,0,2,48,238,1,1.0,2.51,765.0,11.81,14.9,15.0,34.9,0.5,0.0,1.0


In [51]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(fact_trips)

pq.write_to_dataset(
    table,
    root_path       = "data/warehouse/fact_trips",
    partition_cols  = ["pickup_year", "pickup_month"], 
)
dim_geography.to_csv("data/warehouse/dim_geography.csv", index=False)
dim_date.to_csv("data/warehouse/dim_date.csv",           index=False)

print(" All tables saved to data/warehouse")

 All tables saved to data/warehouse


In [52]:
import time

t1 = time.time()
jan_df = pq.read_table(
    "data/warehouse/fact_trips",
    filters=[('pickup_year', '=', 2023), ('pickup_month', '=', 1)]  
).to_pandas()
t2 = time.time()

print(f" Partition-filtered read: {len(jan_df):,} rows in {t2-t1:.3f} seconds")
print(f"\n January 2023 Summary:")
print(f"   Total trips     : {len(jan_df):,}")
print(f"   Avg fare        : ${jan_df['fare_amount'].mean():.2f}")
print(f"   Avg distance    : {jan_df['trip_distance_miles'].mean():.2f} miles")
print(f"   Avg duration    : {jan_df['trip_duration_sec'].mean()/60:.1f} minutes")
print(f"   Total revenue   : ${jan_df['total_amount'].sum():,.2f}")

 Partition-filtered read: 15,428,546 rows in 27.791 seconds

 January 2023 Summary:
   Total trips     : 15,428,546
   Avg fare        : $18.75
   Avg distance    : 3.39 miles
   Avg duration    : 15.8 minutes
   Total revenue   : $424,870,539.53


In [53]:
print("Q1: Top 5 Pickup Boroughs by Trip Count")
print("-" * 40)

q1 = (
    fact_trips
    .merge(dim_geography.rename(columns={'location_id':'pickup_location_id', 'borough':'pickup_borough'}),
           on='pickup_location_id', how='left')
    .groupby('pickup_borough')
    .agg(
        trip_count    = ('trip_id',      'count'),
        avg_fare      = ('fare_amount',  'mean'),
        total_revenue = ('total_amount', 'sum')
    )
    .round(2)
    .sort_values('trip_count', ascending=False)
    .head(5)
)
print(q1)

Q1: Top 5 Pickup Boroughs by Trip Count
----------------------------------------
                trip_count  avg_fare  total_revenue
pickup_borough                                     
Manhattan          2647678     15.02    60449763.85
Queens              277900     51.41    19211462.72
Unknown              39050     27.24     1530816.71
Brooklyn             17605     27.38      593330.58
Bronx                 4044     31.50      144890.01


In [54]:
# ── Q2: Trips by Hour of Day ──────────────────────────────────────────────────
print("\nQ2: Trips by Hour of Day (Rush Hour Analysis)")
print("-" * 40)

q2 = (
    fact_trips
    .groupby('pickup_hour')
    .agg(trip_count=('trip_id','count'), avg_fare=('fare_amount','mean'))
    .round(2)
    .reset_index()
)
peak_hour = q2.loc[q2['trip_count'].idxmax(), 'pickup_hour']
print(q2.to_string(index=False))
print(f"\n Busiest hour: {peak_hour}:00")


Q2: Trips by Hour of Day (Rush Hour Analysis)
----------------------------------------
 pickup_hour  trip_count  avg_fare
           0       82963     20.04
           1       58385     18.14
           2       40899     17.22
           3       26628     18.29
           4       17241     22.84
           5       17503     26.90
           6       42676     22.40
           7       84569     19.15
           8      113890     17.67
           9      127666     17.76
          10      139692     17.84
          11      149988     17.52
          12      165377     17.89
          13      173846     18.59
          14      186231     19.87
          15      191028     19.43
          16      190639     19.70
          17      204060     18.78
          18      210765     17.17
          19      188311     17.75
          20      161990     18.14
          21      157994     18.61
          22      144183     19.51
          23      111935     20.76

 Busiest hour: 18:00
